# Podcast → YouTube バッチパイプライン

## 実行前チェックリスト
- [ ] Google Drive に `podcast-to-youtube/` リポジトリをマウント済み
- [ ] `config/credentials/client_secret.json` を配置済み
- [ ] `config/config.yaml` の `rss.feed_url` を設定済み
- [ ] `assets/artwork.jpg` を配置済み
- [ ] HuggingFace Token を取得済み（pyannote利用規約に同意必須）

**クォータ注意**: YouTube APIは1日最大約6本（デフォルト10,000units/日）

In [ ]:
# ============================================================
# セル1: セットアップ
# ============================================================

# Google Drive をマウント
from google.colab import drive
drive.mount('/content/drive')

# リポジトリのパスを設定（Drive上のパスに合わせて変更してください）
REPO_PATH = '/content/drive/MyDrive/podcast-to-youtube'

import sys, os
sys.path.insert(0, REPO_PATH)
sys.path.insert(0, os.path.join(REPO_PATH, 'scripts'))
os.chdir(REPO_PATH)

# 依存パッケージインストール
!pip install -q -r requirements.txt

# ffmpeg インストール確認
!ffmpeg -version 2>&1 | head -1

print('セットアップ完了')

In [ ]:
# ============================================================
# セル2: 環境変数・認証設定
# ============================================================
import os
from google.colab import userdata

# HuggingFace Token（Colab Secrets または直接入力）
# 方法A: Colab Secrets（推奨）
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN: Colab Secretsから読み込み完了')
except Exception:
    # 方法B: 直接入力
    from getpass import getpass
    os.environ['HF_TOKEN'] = getpass('HuggingFace Token を入力してください: ')

# YouTube OAuth認証
# 初回実行時にブラウザ認証が起動します
# credentials/token.json にキャッシュされるため2回目以降は不要です
from youtube_uploader import get_youtube_client
youtube = get_youtube_client()
print('YouTube OAuth認証: 完了')

In [ ]:
# ============================================================
# セル3: 設定確認・未処理エピソード一覧
# ============================================================
import yaml
from rss_parser import get_pending_episodes, load_progress

with open('config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print(f"RSS URL: {config['rss']['feed_url']}")
print(f"Whisperモデル: {config['processing']['whisper_model']}")
print(f"バッチサイズ: {config['processing']['batch_size']}")
print(f"話者マッピング: {config['podcast']['speakers']}")
print()

pending = get_pending_episodes()
print(f'未処理エピソード数: {len(pending)}')

progress = load_progress()
done = sum(1 for v in progress['episodes'].values() if v.get('status') == 'done')
error = sum(1 for v in progress['episodes'].values() if v.get('status') == 'error')
print(f'処理済み: {done}本 / エラー: {error}本')

print()
print('--- 未処理エピソード一覧（上位10件）---')
for ep in pending[:10]:
    print(f"  [{ep['published_date'][:10]}] {ep['title']}")

In [ ]:
# ============================================================
# セル4: バッチ実行
# ============================================================
# 注意: 1セッションあたり config.processing.batch_size 本を処理します
# クォータ超過(1日6本上限)になると自動停止します

import yaml
from main import run_batch

# バッチ実行（batch_size本処理）
run_batch()

In [ ]:
# ============================================================
# セル5: 進捗確認
# ============================================================
import pandas as pd
from rss_parser import load_progress

progress = load_progress()

rows = []
for guid, info in progress['episodes'].items():
    rows.append({
        'guid': guid[:30] + '...' if len(guid) > 30 else guid,
        'status': info.get('status', 'unknown'),
        'youtube_id': info.get('youtube_id', ''),
        'error_msg': info.get('error_msg', ''),
    })

df = pd.DataFrame(rows)

print('=== 処理状況サマリ ===')
print(df['status'].value_counts().to_string())
print()
print('=== エラー一覧 ===')
errors = df[df['status'] == 'error']
if errors.empty:
    print('エラーなし')
else:
    display(errors[['guid', 'error_msg']])

print()
print('=== 完了一覧（直近10件）===')
done = df[df['status'] == 'done'].tail(10)
display(done[['guid', 'youtube_id']])